<div style="background-color:#000000; padding:20px; border-radius:10px">
    <h1 style="color:#E50914; text-align:center; font-family:Helvetica;">Netflix Content Strategy & Decision Intelligence</h1>
    <p style="color:#ffffff; text-align:center; font-size:16px;">Transforming raw catalogue data into actionable business intelligence to drive future content acquisition and production decisions.</p>
</div>

***
## 🎯 Executive Summary
This Exploratory Data Analysis (EDA) investigates the Netflix dataset to uncover deep insights into content growth, global market penetration, and audience targeting. 

**Key Business Questions Answered:**
1. **Format Strategy:** Should we continue focusing on Movies, or pivot entirely to TV Shows to reduce churn?
2. **Global Expansion:** Which regions represent the next frontier for localized content production?
3. **Target Demographic:** Are we over-indexing on mature audiences at the expense of family households?
4. **Genre Investment:** Where should the content acquisition budget be allocated next quarter?

In [1]:
# Import required libraries for premium data visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Custom Netflix-inspired aesthetic palette
netflix_red = '#E50914'
netflix_dark = '#141414'
netflix_grey = '#b3b3b3'

sns.set_style("darkgrid", {"axes.facecolor": "#f5f5f5"})
plt.rcParams.update({'text.color': '#333333', 'axes.labelcolor': '#333333'})

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'wordcloud'

### Data Engineering & Cleaning

In [2]:
df = pd.read_csv('netflix_titles.csv')

# Pre-processing
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna('Unknown', inplace=True)

C:\Users\Aesha\AppData\Local\Temp\ipykernel_13456\2046182297.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['country'].fillna('Unknown', inplace=True)
C:\Users\Aesha\AppData\Local\Temp\ipykernel_13456\2046182297.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exampl

## 1. Format Strategy: Movies vs. TV Shows
Understanding the volume of content types helps dictate budget allocation. TV Shows naturally have higher retention rates (binge-watching), whereas movies drive quick acquisitions.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#ffffff')

# Pie chart
type_counts = df['type'].value_counts()
ax[0].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=90, 
        colors=[netflix_red, '#4f4f4f'], explode=(0.05, 0), textprops={'fontsize': 14, 'color': 'black', 'weight': 'bold'})
ax[0].set_title('Overall Content Format Split', fontsize=18, weight='bold')

# Trend chart
trend_df = df.groupby(['year_added', 'type']).size().unstack().fillna(0)
trend_df = trend_df[trend_df.index >= 2010]
ax[1].plot(trend_df.index, trend_df['Movie'], color=netflix_red, linewidth=3, label='Movie')
ax[1].plot(trend_df.index, trend_df['TV Show'], color='#4f4f4f', linewidth=3, label='TV Show')
ax[1].fill_between(trend_df.index, trend_df['Movie'], color=netflix_red, alpha=0.1)
ax[1].fill_between(trend_df.index, trend_df['TV Show'], color='#4f4f4f', alpha=0.1)
ax[1].set_title('Acquisition Trend Over Time', fontsize=18, weight='bold')
ax[1].set_ylabel('Number of Titles added', fontsize=14)
ax[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

💡 **Strategic Decision:** While Movies dominate the overall catalog (nearly 70%), the trend chart reveals a massive drop-off post-2019. 
> **Recommendation:** Accelerate investment in multi-season TV Shows. They provide higher Return on Investment (ROI) by keeping subscribers engaged month-over-month, reducing churn compared to one-off movie viewings.

## 2. Global Expansion: Identifying Underserved Markets
Where is our content coming from, and where should we establish our next regional production hub?

In [ ]:
plt.figure(figsize=(14, 8))
top_countries = df['country'].str.split(', ').explode().value_counts()[1:11] # Skip 'Unknown'

sns.barplot(y=top_countries.index, x=top_countries.values, palette="Reds_r", edgecolor='black')
plt.title('Top 10 Content Producing Countries', fontsize=18, weight='bold')
plt.xlabel('Total Titles', fontsize=14)
plt.ylabel('Country', fontsize=14)

# Add data labels
for i, v in enumerate(top_countries.values):
    plt.text(v + 20, i + 0.1, str(v), color='black', fontweight='bold', fontsize=12)

plt.show()

💡 **Strategic Decision:** The United States and India are absolute powerhouses, proving the success of our local-to-global strategy.
> **Recommendation:** South Korea and Japan are climbing the ranks, indicating massive global appetite for K-Dramas and Anime. We should increase production budgets for the APAC region to capture the massive Southeast Asian demographic.

## 3. Genre Investment: The Content Tag Cloud
What genres occupy the most space in our ecosystem?

In [ ]:
text = ' '.join(df['listed_in'].dropna())
wordcloud = WordCloud(width=1200, height=600, background_color=netflix_dark, 
                      colormap='Reds', max_words=100, contour_width=3).generate(text)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Content Genre Dominance', fontsize=22, weight='bold', color=netflix_red)
plt.show()

💡 **Strategic Decision:** "International Movies", "Dramas", and "Comedies" are highly saturated.
> **Recommendation:** To stand out in a crowded streaming market, we should pivot toward niche, high-engagement genres like "Sci-Fi & Fantasy" or "Docuseries", which have lower volume but historically high completion rates and critical acclaim.

## 4. Audience Targeting: Maturity Rating Distribution
Are we appealing to the whole family?

In [ ]:
plt.figure(figsize=(14, 6))
rating_order = df['rating'].value_counts().index
sns.countplot(x='rating', data=df, order=rating_order, palette='dark:red_r')
plt.title('Audience Maturity Rating Profile', fontsize=18, weight='bold')
plt.xlabel('Rating Category', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(rotation=45)
plt.show()

💡 **Strategic Decision:** The overwhelming majority of our content is rated TV-MA (Mature Audiences).
> **Recommendation:** We are highly exposed to churn from family households who may migrate to competitors like Disney+. We must launch a strategic initiative to acquire and produce more **TV-Y, TV-Y7, and TV-PG** content to solidify our position as a multi-generational household staple.

***
<h2 align="center" style="color:#E50914">Final Conclusion</h2>

By shifting budget from standalone movies to **serialized TV shows**, increasing localized production in **APAC (South Korea/Japan)**, and consciously acquiring **Family-Friendly (TV-Y/PG)** content, Netflix can aggressively combat churn and capture new global subscribers.

*For live, interactive filtering of this data, please launch the executive dashboard by running `streamlit run app.py`.*